In [ ]:
!pip install -q transformers accelerate peft trl bitsandbytes datasets scikit-learn
!git clone https://github.com/<YOUR_REPO>/troke /content/troke  # TODO: replace <YOUR_REPO>
import sys; sys.path.insert(0, "/content/troke")

In [ ]:
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get("HF_TOKEN"))


In [ ]:
import torch, yaml
from finetune.train import apply_hardware_overrides

def detect_hardware():
    if not torch.cuda.is_available(): return "rtx4060"
    name = torch.cuda.get_device_name(0).lower()
    if "a100" in name: return "a100"
    if "t4" in name: return "t4"
    return "rtx4060"

hw = detect_hardware()
print(f"Hardware: {hw}")

with open("/content/troke/finetune/config.yaml") as f:
    cfg = yaml.safe_load(f)

cfg = apply_hardware_overrides(cfg, hw)
print(f"Batch size: {cfg['training']['per_device_train_batch_size']}, 4bit: {cfg['quantization']['load_in_4bit']}")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
CKPT_DIR = "/content/drive/MyDrive/troke_checkpoints/dermatology-isic2019"
cfg["model"]["adapter_output_dir"] = CKPT_DIR


In [ ]:
import glob
from finetune.train import (
    load_model_and_processor, apply_lora, prepare_datasets,
    build_training_args, DermatologyCollator
)
from trl import SFTTrainer

processor, model = load_model_and_processor(
    cfg["model"], cfg["quantization"], cfg["quantization"]["load_in_4bit"]
)
model = apply_lora(model, cfg["lora"])
model.print_trainable_parameters()

train_ds, val_ds = prepare_datasets(cfg, processor)
checkpoints = sorted(
    glob.glob(f"{CKPT_DIR}/checkpoint-*"),
    key=lambda p: int(p.rsplit("-", 1)[-1])
)
resume_from = checkpoints[-1] if checkpoints else None
print(f"Resuming from: {resume_from}")

trainer = SFTTrainer(
    model=model, args=build_training_args(cfg["training"], CKPT_DIR),
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=DermatologyCollator(processor, cfg["training"]["max_seq_length"]),
)
trainer.train(resume_from_checkpoint=resume_from)
model.save_pretrained(CKPT_DIR)
processor.save_pretrained(CKPT_DIR)
print("Done. Push to Hub with: model.push_to_hub('your-org/medgemma-dermatology')")